# EDA On `players.csv` Dataset

This dataset has 1 row per player in the rookie cohort. 

It very simply describes them with attributes like position, birthday, draft pick, etc.

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    RAW_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

In [ ]:
# ==========================================
# Load & Clean Data
# ==========================================

players_df = (
    pl.read_csv(
        str(RAW_DATA_DIR / "players.csv"), 
        # Change "NA" to None
        null_values = {
            "draft_round": "NA", 
            "draft_pick_within_round": "NA", 
            "draft_overall_pick": "NA"
        }
    )
    # ---------------------------------------
    # Change draft_round to categorical variables with groups: {1, 2, 3, 4, 5, 6, 7, UDFA}
    # ---------------------------------------
    # Change to string
    .with_columns(draft_round = pl.col("draft_round").cast(pl.String))
    # Set missing value to UDFA
    .with_columns(
        draft_round = pl.when(pl.col("draft_round").is_null())
                        .then(pl.lit("UDFA"))
                        .otherwise(pl.col("draft_round"))
    )
    # Change to categorical variable
    .with_columns(draft_round = pl.col("draft_round").cast(pl.Categorical))
)

print("Players Dataset:")
print(f"     - Dimensions: {players_df.shape}")
players_df

In [ ]:
players_df.filter(pl.col("nfl_id") == 55970)

In [ ]:
# =======================================
# Missing Data
# =======================================

# We see below that missing data is not an issue, all missingness from
# draft_pick_within_round and draft_overall_pick is expected because of UDFAs

players_df.select(pl.all().is_null().sum())

In [ ]:
# =======================================
# nfl_id / display_name
# =======================================

print(f"  - Number of Unique Player IDs:   {players_df.select("nfl_id").n_unique()}")
print(f"  - Number of Unique Player Names: {players_df.select("display_name").n_unique()}")
print(f"     - NOTE: The fact that there are 508 unique names is because some people share the same name, not a data error.")
print(f"  - Number of Unique nfl_id/display_name Combinations: {players_df.select("nfl_id", "display_name").n_unique()}")

In [ ]:
# ========================================
# draft_year
# ========================================

players_df.group_by("draft_year").agg(count = pl.len()).sort("draft_year")

In [ ]:
# =======================================
# nfl_position
# =======================================

print(f"  - Number of Unique Positions in Raw Data: {players_df.select("nfl_position").n_unique()}")
position_counts = players_df.group_by("nfl_position").agg(count = pl.len()).sort("nfl_position")

fig = px.bar(
    position_counts, 
    x = "nfl_position", 
    y = "count", 
    title = "Position Counts Among Rookie Cohort"
)

fig.show()

In [ ]:
# ====================================
# draft_round
# ====================================

players_df.group_by("draft_round").agg(count = pl.len()).with_columns(perc = (100 * pl.col("count") / pl.sum("count")).round(1)).sort("draft_round")